# D14 · Series temporales (pronóstico)

**Formación Pública — Bootcamp de Ciencia de Datos para funcionarios públicos**
Bloque avanzado · IA aplicada (opcional) · Se ejecuta en Google Colab.

---

## ¿Qué vas a lograr?

Hasta ahora trabajaste con datos donde cada fila es independiente: un proveedor, un organismo, una orden de compra. En este módulo cambia algo fundamental: **el tiempo importa**. Vas a trabajar con una **serie temporal** —el gasto público de Chile año a año— y vas a aprender a hacer lo que más se le pide a un científico de datos del Estado: **mirar hacia adelante**. ¿Cuánto comprará el Estado el próximo año? ¿La tendencia sube o se aplana? ¿Mi estimación es confiable o me estoy engañando?

Al terminar serás capaz de:

- Entender qué es una serie temporal y en qué se diferencia de una tabla común.
- Cargar una serie real, graficarla y **describir su tendencia**.
- Medir el crecimiento período a período.
- Construir **tres pronósticos base** (ingenuo, crecimiento medio y tendencia lineal).
- **Validar** un pronóstico con un método honesto (hold-out / backtest) y elegir el mejor.

**Competencia de salida:** dado un registro temporal real del Estado, producir un pronóstico defendible y justificar por qué es confiable.

**Entregable:** que las **4 celdas de chequeo** de este cuaderno muestren ✅.

> **Requisito previo:** haber completado el núcleo (M0–D13). Damos por sabido pandas, numpy y matplotlib; lo nuevo aquí es la *forma de pensar* en el tiempo.


## 1. ¿Qué es una serie temporal?

Una **serie temporal** es una secuencia de valores **ordenados en el tiempo** y medidos a intervalos regulares: el gasto de cada año, las ventas de cada mes, la temperatura de cada día. La palabra clave es *ordenados*: el orden no es un detalle, es la información.

Piénsalo así. Si te doy las edades de 50 funcionarios, da igual en qué orden las leas: la media es la misma. Pero si te doy el gasto del Estado de 2019 a 2025 y te lo **desordeno**, perdiste lo más valioso —si venía subiendo o bajando, si hubo un salto en pandemia, hacia dónde apunta. En una serie temporal, **la fila de al lado importa**.

### Las tres piezas de una serie

Casi cualquier serie temporal se puede pensar como la suma de tres ingredientes:

1. **Tendencia (trend):** hacia dónde va a largo plazo. ¿Sube, baja, se aplana? El gasto público chileno, por ejemplo, lleva años subiendo.
2. **Estacionalidad (seasonality):** patrones que se **repiten** en ciclos fijos. En compras públicas es típico que diciembre dispare el gasto (cierre de presupuesto) y enero lo desplome. Pero —ojo— la estacionalidad **solo se ve con datos sub-anuales** (meses, trimestres). Con datos **anuales**, como los de hoy, la estacionalidad queda "promediada hacia adentro" de cada año y no la podemos observar. Lo decimos con todas sus letras para que no te confundas: **en este módulo trabajamos la tendencia, no la estacionalidad.**
3. **Ruido (noise):** lo que queda, las fluctuaciones que no explica ni la tendencia ni la estacionalidad. Ningún pronóstico lo elimina; un buen pronóstico solo no lo confunde con señal.

### ¿Por qué pronosticar?

Porque casi toda decisión pública mira hacia adelante. Cuánto presupuesto pedir, cuántos contratos esperar, si una tendencia de gasto es sostenible. Un pronóstico no es una bola de cristal: es **una estimación con supuestos explícitos**. Y la habilidad clave no es "acertar", es **saber cuánto puedes confiar** en lo que estimaste. A eso le dedicamos el final del módulo.


## 2. Preparación

Cargamos las librerías de siempre. Nada nuevo que instalar.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Para mostrar los montos grandes sin notación científica
pd.set_option("display.float_format", lambda x: f"{x:,.0f}")
print("Librerías listas ✔")

## 3. Los datos: gasto público de Chile, año a año

Estos datos son **reales** y vienen de **ChileCompra / MercadoPúblico** (indicador de monto acumulado del Estado). Cada fila es un año: el **gasto total en compras públicas** (en pesos chilenos) y la **cantidad de órdenes de compra** emitidas.

Un detalle importante que vas a usar más adelante: el **2026 está incompleto** (es un año "parcial", todavía en curso al momento de tomar el dato). Por eso lo marcamos aparte: no se puede comparar de igual a igual con un año cerrado.

Ejecuta la celda. La variable `df` queda disponible para todo el cuaderno.


In [ ]:
# ── Preparación del entorno (ejecuta esta celda primero) ──────────────────────
import os
import urllib.request
import pandas as pd

# Si el archivo no existe localmente (ej: en Colab), intentamos descargarlo desde GitHub
if not os.path.exists("gasto_anual.csv"):
    try:
        url = "https://raw.githubusercontent.com/formacionpublica-bootcamp/bootcamp/main/B7-series-temporales/gasto_anual.csv"  # Reemplazar por la URL real del repositorio al publicar
        urllib.request.urlretrieve(url, "gasto_anual.csv")
        print("gasto_anual.csv descargado automáticamente.")
    except Exception:
        print("Aviso: No se pudo descargar gasto_anual.csv automáticamente. Si estás en Colab, asegúrate de subirlo manualmente.")

# Datos reales de ChileCompra (gasto anual en compras públicas, 2019-2025).
# Obtenidos a través de la Dirección de Compras y Contratación Pública (ChileCompra).
df_gasto = pd.read_csv("gasto_anual.csv")

# Convertimos a formato de lista de dicts para mantener la variable DATOS y no
# romper la compatibilidad con el resto del notebook y las celdas de chequeo.
DATOS = df_gasto.to_dict(orient="records")

df = pd.DataFrame(DATOS)
df

## Ejercicio 1 — Construir y graficar la serie

Una tabla con una columna de años todavía no es una "serie temporal" lista para trabajar. El primer paso siempre es el mismo: **dejar el tiempo como índice** y quedarnos solo con los años **completos** (los parciales distorsionan cualquier análisis).

Tu tarea:

1. A partir de `df`, quédate solo con las filas donde `parcial` es `False`.
2. Construye una **Serie de pandas** llamada `gasto`, **indexada por el año** (`anio`), con los valores de `gasto_clp`.
3. Grafícala con `gasto.plot(marker="o")` para verla.

> 💡 **Pista:** `df[df["parcial"] == False]` filtra las filas; luego `.set_index("anio")["gasto_clp"]` te deja la Serie indexada por año.

> ⚠️ **Error típico:** dejar dentro el 2026 parcial. Una serie con un año "a medias" hace que la tendencia parezca caer de golpe al final. Filtra **antes** de graficar.


In [ ]:
# TODO: construye la Serie `gasto` (solo años completos, indexada por año)
gasto = None   # <-- reemplaza esto

# Una vez que `gasto` sea una Serie, esta línea la grafica:
if gasto is not None:
    gasto.plot(marker="o", title="Gasto público de Chile en compras (CLP)")
    plt.xlabel("Año"); plt.ylabel("Gasto (CLP)"); plt.grid(True); plt.show()

In [ ]:
# Celda de chequeo — Ejercicio 1
def _chequeo_1():
    esperado = (pd.DataFrame(DATOS)
                  .query("parcial == False")
                  .set_index("anio")["gasto_clp"])
    try:
        g = gasto
    except NameError:
        print("❌ Aún no existe `gasto`. Completa el TODO de arriba."); return
    if g is None:
        print("❌ `gasto` sigue en None. Construye la Serie con los años completos."); return
    try:
        assert isinstance(g, pd.Series), "`gasto` debe ser una Serie de pandas, no un DataFrame ni una lista."
        assert list(g.index) == list(esperado.index), \
            f"El índice debería ser los años completos {list(esperado.index)}. ¿Incluiste el 2026 parcial por error?"
        assert np.allclose(g.values.astype(float), esperado.values.astype(float)), \
            "Los valores no coinciden con el gasto real. Revisa que uses la columna `gasto_clp`."
        print("✅ Correcto: serie de 7 años (2019–2025), lista para analizar.")
        print(f"   Gasto 2019: {g.loc[2019]:,.0f}  →  Gasto 2025: {g.loc[2025]:,.0f} CLP")
    except AssertionError as e:
        print(f"❌ Casi. Pista: {e}")

_chequeo_1()

## Ejercicio 2 — Medir la tendencia: crecimiento año a año

Ver que la línea "sube" es cualitativo. Para describir una tendencia de verdad necesitamos **números**: ¿cuánto crece de un año al siguiente, en porcentaje?

La **tasa de crecimiento** de un año respecto del anterior es:

$$\text{crecimiento}_t = \left(\frac{\text{valor}_t}{\text{valor}_{t-1}} - 1\right) \times 100$$

pandas tiene esto resuelto: `gasto.pct_change()` calcula la variación relativa fila a fila (el primer año queda en `NaN` porque no tiene año previo con qué compararse —es normal).

Tu tarea:

1. Calcula `crecimiento = gasto.pct_change() * 100` (en porcentaje).
2. Observa el resultado: vas a ver años de ~10% y un salto fuerte en 2024.

> 💡 **Pista:** `pct_change()` devuelve fracciones (0.10 = 10%). Multiplica por 100 para leerlo como porcentaje.

> ⚠️ **Error típico:** asustarse por el `NaN` del primer año. No es un error: el 2019 no tiene "año anterior" en la serie.


In [ ]:
# TODO: calcula la tasa de crecimiento año a año, en porcentaje
crecimiento = None   # <-- reemplaza esto

crecimiento

In [ ]:
# Celda de chequeo — Ejercicio 2
def _chequeo_2():
    base = (pd.DataFrame(DATOS).query("parcial == False")
              .set_index("anio")["gasto_clp"])
    esperado = base.pct_change() * 100
    try:
        c = crecimiento
    except NameError:
        print("❌ Aún no existe `crecimiento`. Completa el TODO."); return
    if c is None:
        print("❌ `crecimiento` sigue en None. Usa pct_change()."); return
    try:
        assert isinstance(c, pd.Series), "`crecimiento` debe ser una Serie."
        a = c.dropna().values.astype(float)
        b = esperado.dropna().values.astype(float)
        assert len(a) == len(b), "La serie de crecimiento tiene un largo inesperado."
        assert np.allclose(a, b, rtol=1e-6), \
            "Los porcentajes no cuadran. ¿Multiplicaste por 100? ¿Usaste pct_change() sobre `gasto`?"
        print("✅ Correcto: tasas de crecimiento calculadas.")
        print(f"   Crecimiento 2024: {c.loc[2024]:.1f}%   ·   Crecimiento 2025: {c.loc[2025]:.1f}%")
    except AssertionError as e:
        print(f"❌ Casi. Pista: {e}")

_chequeo_2()

## Ejercicio 3 — Tres pronósticos base (backtest sobre 2025)

Aquí viene la idea más importante del módulo: **nunca evalúes un pronóstico con los datos que usaste para construirlo.** Sería como corregir tu propia prueba con el solucionario al lado. La técnica honesta se llama **hold-out** (o *backtest*): escondes el último dato real, pronosticas como si no lo conocieras, y **recién después** lo destapas para ver qué tan cerca estuviste.

Vamos a esconder el **2025**. Entrenamos solo con **2019–2024** y pronosticamos 2025 con tres métodos clásicos, de más simple a menos:

- **Ingenuo (naïve):** "el próximo año será igual al último". Pronóstico = valor de 2024. Parece tonto, pero es la **vara mínima**: si un método elaborado no le gana al ingenuo, no sirve.
- **Crecimiento medio:** "el próximo año crecerá como vino creciendo en promedio". Pronóstico = valor 2024 × (1 + crecimiento medio histórico).
- **Tendencia lineal:** ajustamos una recta a los puntos y la extendemos un año. `np.polyfit(años, valores, 1)` da la recta; `np.polyval` la evalúa en 2025.

Tu tarea: completa el diccionario `pronosticos_2025` con las tres estimaciones.

> 💡 **Pista:** `train = gasto.loc[2019:2024]`. El crecimiento medio es `train.pct_change().mean()`. Para la recta: `coef = np.polyfit(train.index, train.values, 1)` y luego `np.polyval(coef, 2025)`.

> ⚠️ **Error típico:** incluir el 2025 en `train`. Si lo haces, el "pronóstico" ya vio la respuesta y la validación deja de tener sentido.


In [ ]:
# TODO: entrena con 2019–2024 y pronostica 2025 con los tres métodos
train = None              # <-- la serie 2019–2024
pron_ingenuo = None       # <-- último valor (2024)
pron_crec_medio = None    # <-- 2024 * (1 + crecimiento medio)
pron_lineal = None        # <-- recta ajustada, evaluada en 2025

pronosticos_2025 = {
    "ingenuo": pron_ingenuo,
    "crecimiento_medio": pron_crec_medio,
    "tendencia_lineal": pron_lineal,
}
pronosticos_2025

In [ ]:
# Celda de chequeo — Ejercicio 3
def _chequeo_3():
    base = (pd.DataFrame(DATOS).query("parcial == False")
              .set_index("anio")["gasto_clp"]).astype(float)
    tr = base.loc[2019:2024]
    esp = {
        "ingenuo": tr.loc[2024],
        "crecimiento_medio": tr.loc[2024] * (1 + tr.pct_change().mean()),
        "tendencia_lineal": float(np.polyval(np.polyfit(tr.index, tr.values, 1), 2025)),
    }
    try:
        p = pronosticos_2025
    except NameError:
        print("❌ Aún no existe `pronosticos_2025`. Completa el TODO."); return
    if not isinstance(p, dict) or any(p.get(k) is None for k in esp):
        print("❌ Faltan pronósticos. Completa los tres métodos en el diccionario."); return
    try:
        for k in esp:
            assert np.isclose(float(p[k]), esp[k], rtol=1e-6), \
                f"El método '{k}' no coincide con el valor esperado. Revisa la fórmula."
        print("✅ Correcto: tres pronósticos para 2025 (entrenando solo con 2019–2024).")
        for k, v in esp.items():
            print(f"   {k:18s}: {v:,.0f} CLP")
    except AssertionError as e:
        print(f"❌ Casi. Pista: {e}")

_chequeo_3()

## Ejercicio 4 — Evaluar, elegir el mejor y pronosticar 2026

Ahora **destapamos el 2025 real** y medimos qué tan lejos quedó cada pronóstico. Usamos el **error absoluto**: la distancia (sin signo) entre lo pronosticado y lo real.

$$\text{error} = |\,\text{pronóstico} - \text{real}\,|$$

El método con **menor error** es el que mejor describió esta serie. Y con ese método —ya validado— damos el paso que de verdad importa: **pronosticar 2026**.

Tu tarea:

1. Toma el gasto **real de 2025** (está en `gasto.loc[2025]`).
2. Construye `errores`: un diccionario con el error absoluto de cada método (mismas claves que `pronosticos_2025`).
3. Guarda en `mejor_metodo` la clave con **menor** error.
4. Con ese método, calcula `pronostico_2026` (usando ahora **toda** la serie 2019–2025).

> 💡 **Pista:** `min(errores, key=errores.get)` te da la clave del menor error. Para el pronóstico 2026 con crecimiento medio: `gasto.loc[2025] * (1 + gasto.pct_change().mean())`.

> ⚠️ **Tentación a evitar:** comparar tu pronóstico 2026 con el dato parcial de 2026 ($6,19 billones). **No son comparables**: ese número es un año a medias, no un año cerrado. Compararlos te haría creer que te equivocaste por un factor de 3, cuando en realidad estás mirando media película.


In [ ]:
# TODO: evalúa contra el 2025 real, elige el mejor método y pronostica 2026
real_2025 = None        # <-- gasto real de 2025
errores = None          # <-- dict: método -> error absoluto vs real_2025
mejor_metodo = None     # <-- clave del menor error
pronostico_2026 = None  # <-- con el mejor método, sobre toda la serie 2019–2025

print("Errores vs 2025 real:", errores)
print("Mejor método:", mejor_metodo)
print("Pronóstico 2026:", f"{pronostico_2026:,.0f} CLP" if pronostico_2026 else None)

In [ ]:
# Celda de chequeo — Ejercicio 4
def _chequeo_4():
    base = (pd.DataFrame(DATOS).query("parcial == False")
              .set_index("anio")["gasto_clp"]).astype(float)
    tr = base.loc[2019:2024]
    pron = {
        "ingenuo": tr.loc[2024],
        "crecimiento_medio": tr.loc[2024] * (1 + tr.pct_change().mean()),
        "tendencia_lineal": float(np.polyval(np.polyfit(tr.index, tr.values, 1), 2025)),
    }
    real = base.loc[2025]
    esp_err = {k: abs(v - real) for k, v in pron.items()}
    esp_mejor = min(esp_err, key=esp_err.get)
    esp_2026 = base.loc[2025] * (1 + base.pct_change().mean())
    try:
        e, m, p26 = errores, mejor_metodo, pronostico_2026
    except NameError:
        print("❌ Faltan variables (`errores`, `mejor_metodo` o `pronostico_2026`). Completa el TODO."); return
    if e is None or m is None or p26 is None:
        print("❌ Aún hay variables en None. Completa los cuatro pasos."); return
    try:
        for k in esp_err:
            assert k in e and np.isclose(float(e[k]), esp_err[k], rtol=1e-6), \
                f"El error de '{k}' no coincide. ¿Usaste el valor absoluto contra el 2025 real?"
        assert m == esp_mejor, \
            f"El mejor método debería ser '{esp_mejor}' (el de menor error)."
        assert np.isclose(float(p26), esp_2026, rtol=1e-6), \
            "El pronóstico 2026 no coincide. Usa el crecimiento medio sobre TODA la serie 2019–2025."
        print(f"✅ Correcto: el método ganador es '{m}'.")
        print(f"   Pronóstico 2026 (año completo): {p26:,.0f} CLP")
        parcial_2026 = next(d["gasto_clp"] for d in DATOS if d["anio"] == 2026)
        print(f"   Recordatorio: el 2026 parcial ({parcial_2026:,.0f}) NO es comparable: es un año a medias.")
    except AssertionError as e2:
        print(f"❌ Casi. Pista: {e2}")

_chequeo_4()

## Cierre — Lo que te llevas

- Una **serie temporal** ordena los datos en el tiempo, y ese orden es información: revela **tendencia**, **estacionalidad** y **ruido**.
- Con datos **anuales** describimos la **tendencia**; la estacionalidad necesita datos **mensuales** (será tema de otro módulo cuando tengamos esa serie).
- Existen pronósticos **base** que cualquiera debería probar primero: el **ingenuo** es la vara mínima a vencer.
- La regla de oro: **valida con datos que el modelo no vio** (hold-out / backtest). Sin validación, un pronóstico es solo una opinión con decimales.
- En esta serie, el **crecimiento medio** le ganó al ingenuo y a la recta: el gasto público venía creciendo de forma compuesta, y ese patrón describió mejor el futuro cercano.

### Para llevar al trabajo
La próxima vez que te pidan "una proyección" de cualquier indicador que avance en el tiempo —gasto, demanda, casos, solicitudes—, ya tienes el método: grafica, mide el crecimiento, prueba varios pronósticos simples y **valida escondiendo el último período**. Es simple, es honesto y es defendible frente a quien te pregunte "¿y cómo sé que esto es confiable?".

---
*Formación Pública · Contenido bajo licencia CC BY 4.0. Datos: ChileCompra / MercadoPúblico.*
